In [2]:
import torch, subprocess, sys, os, platform
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi -L || echo "No GPU listed (Colab may still have T4 or L4)"


Torch: 2.8.0+cu126
CUDA available: True
GPU 0: NVIDIA L4 (UUID: GPU-ff10de7f-6433-b8a2-c879-18468a99e292)


In [3]:
!pip -q install "unsloth>=2025.9.0" "transformers>=4.45.0" "datasets>=2.20.0" "accelerate>=1.0.0" "trl>=0.9.6" "bitsandbytes>=0.43.0" "peft>=0.13.0" "evaluate" "scikit-learn" -U


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.3/351.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 122.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 141.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
from getpass import getpass
HF_TOKEN = ""  # paste your token or leave blank to skip pushing
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)


In [5]:
BASE_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"  # tiny instruct model
# If you used a different dataset in Colab1, swap here:
DATASET_NAME = "yahma/alpaca-cleaned"  # simple instruction-following set
SPLIT = "train"                        # or "train[:5000]" to subset while testing

MAX_SEQ_LEN = 1024   # plenty for Alpaca-style tasks
OUTPUT_DIR  = "smollm2_135m_lora_alpaca"
MERGED_DIR  = f"{OUTPUT_DIR}_merged"


In [6]:
from datasets import load_dataset

ds = load_dataset(DATASET_NAME, split=SPLIT)

def to_chat(sample):
    inst = sample.get("instruction","").strip()
    inp  = sample.get("input","").strip()
    tgt  = sample.get("output","").strip()
    # Build a single-turn chat
    if inp:
        user_text = f"{inst}\n\nInput:\n{inp}"
    else:
        user_text = inst

    return {
        "messages": [
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": tgt},
        ]
    }

chat_ds = ds.map(to_chat, remove_columns=ds.column_names)
chat_ds = chat_ds.shuffle(seed=3407)
len(chat_ds), chat_ds[0]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

(51760,
 {'messages': [{'content': 'List 5 popular dishes in US.', 'role': 'user'},
   {'content': '1. Hamburger: A classic American dish consisting of a beef patty served on a bun, often topped with cheese and various toppings such as lettuce, tomatoes, onions, and condiments like ketchup and mustard.\n\n2. Macaroni & Cheese: A comforting dish made from elbow macaroni mixed with a creamy cheese sauce and often baked until golden brown and bubbly.\n\n3. Fried Chicken: A southern staple, fried chicken is typically made from pieces of chicken that are coated in a seasoned batter and deep-fried until crispy and golden brown.\n\n4. Pizza: Pizza is a beloved food in the United States, with toppings ranging from classic pepperoni and cheese to more unique combinations like pineapple and ham.\n\n5. Apple Pie: A classic American dessert made from a flaky pastry crust filled with a sweet and tart apple filling, often served with whipped cream or a scoop of vanilla ice cream.',
    'role': 'assi

In [7]:
import torch
from unsloth import FastLanguageModel

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = BASE_MODEL,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = dtype,
    load_in_4bit    = False,  # True works too; 135M fits easily so keep False for speed
)

# Apply the model's native chat template later during tokenization
assert tokenizer.chat_template is not None, "Tokenizer should provide a chat template for instruct models."

# Attach LoRA adapters (no full_finetuning!)
model = FastLanguageModel.get_peft_model(
    model,
    r                           = 16,
    lora_alpha                  = 16,
    lora_dropout                = 0.0,
    target_modules              = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],  # common safe set
    use_gradient_checkpointing  = True,
    random_state                = 3407,
    max_seq_length              = MAX_SEQ_LEN,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

HuggingFaceTB/SmolLM2-135M-Instruct does not have a padding token! Will use pad_token = <|endoftext|>.


Unsloth 2025.11.2 patched 30 layers with 30 QKV layers, 30 O layers and 30 MLP layers.


In [8]:
from functools import partial

def tokenize_chat(example, tokenizer, max_len):
    # Convert messages → prompt + labels using the tokenizer’s chat template
    prompt = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    toks   = tokenizer(prompt, truncation=True, max_length=max_len, padding=False)
    # For SFT we need labels; simple next-token prediction—copy input_ids
    toks["labels"] = toks["input_ids"].copy()
    return toks

tokenize_fn = partial(tokenize_chat, tokenizer=tokenizer, max_len=MAX_SEQ_LEN)
tokenized_ds = chat_ds.map(tokenize_fn, remove_columns=chat_ds.column_names)
tokenized_ds = tokenized_ds.filter(lambda x: len(x["input_ids"]) < MAX_SEQ_LEN)  # keep within budget


Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

Filter:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [11]:
# === FAST MODE SFT (quicker runs) ===
import torch, math, os
from trl import SFTTrainer
from transformers import TrainingArguments

# 1) Trim dataset + context length for speed
#    (adjust slices if you want slightly longer runs)
if "tokenized_ds" in globals():
    fast_ds = tokenized_ds.select(range(min(2000, len(tokenized_ds))))  # ~2k samples
else:
    fast_ds = train_dataset.select(range(min(2000, len(train_dataset))))  # fallback

FAST_MAX_SEQ_LEN = min(512, MAX_SEQ_LEN)  # cut sequence length to 512

# 2) Use faster/more memory-friendly optimizer if available
#    - "adamw_bnb_8bit" (bitsandbytes) is usually fastest on Colab/Kaggle
#    - fallback to adamw_torch if bnb not present
try:
    import bitsandbytes as bnb  # noqa
    fast_optim = "adamw_bnb_8bit"
except Exception:
    fast_optim = "adamw_torch"

# 3) Batch + steps: smaller grads, fewer steps, less logging/saving
PER_DEVICE_BATCH = 8                      # bump this if VRAM allows; lower to 4 if OOM
GRAD_ACCUM       = 2                      # small accumulation keeps steps snappy
MAX_STEPS        = 200                    # ~a few minutes on T4/L4
LOG_STEPS        = 50
SAVE_STEPS       = 10_000                 # effectively disables periodic saves
SAVE_TOTAL_LIMIT = 1

# 4) Mixed precision & checkpointing trade-offs
use_bf16 = (dtype == torch.bfloat16)
use_fp16 = (dtype == torch.float16)

# gradient_checkpointing=False is faster but uses more VRAM; set True if OOM
USE_GC = False

# 5) Dataloader speedups
NUM_WORKERS = 2
PIN_MEMORY  = True

args = TrainingArguments(
    output_dir                   = OUTPUT_DIR,
    learning_rate                = 3e-4,               # slightly higher LR for short runs
    lr_scheduler_type            = "cosine",
    warmup_ratio                 = 0.03,
    weight_decay                 = 0.0,
    per_device_train_batch_size  = PER_DEVICE_BATCH,
    gradient_accumulation_steps  = GRAD_ACCUM,
    max_steps                    = MAX_STEPS,
    logging_steps                = LOG_STEPS,
    save_steps                   = SAVE_STEPS,
    save_total_limit             = SAVE_TOTAL_LIMIT,
    bf16                         = use_bf16,
    fp16                         = use_fp16,
    optim                        = fast_optim,
    gradient_checkpointing       = USE_GC,
    report_to                    = "none",
    dataloader_num_workers       = NUM_WORKERS,
    dataloader_pin_memory        = PIN_MEMORY,
)

# Optional: enable slightly faster matmul (safe on most GPUs)
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

trainer = SFTTrainer(
    model              = model,           # your LoRA-wrapped or base model
    tokenizer          = tokenizer,
    train_dataset      = fast_ds,
    max_seq_length     = FAST_MAX_SEQ_LEN,
    packing            = True,            # keep this on for throughput
    dataset_text_field = None,            # already tokenized/mapped upstream
    args               = args,
)

print("🚀 Fast training starting…")
trainer.train()
print("🏁 Fast training done.")


The model is already on multiple devices. Skipping the move to device specified in `args`.


🚀 Fast training starting…


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 2 | Total steps = 200
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 4,884,480 of 139,399,488 (3.50% trained)


Step,Training Loss
50,1.371800
100,1.368200
150,1.344100
200,1.334800


🏁 Fast training done.


In [12]:
# === Robust saver for LoRA adapter + merged model (handles trainer/model/policy + unsloth versions) ===
import os, sys

# 0) Figure out model handles you might have
MODEL_OBJ = None
if "trainer" in globals() and hasattr(trainer, "model") and trainer.model is not None:
    MODEL_OBJ = trainer.model
elif "policy" in globals():
    MODEL_OBJ = policy
elif "model" in globals():
    MODEL_OBJ = model
else:
    raise SystemExit("❌ No model found. Expected one of: trainer.model, policy, or model.")

# 1) Ensure dirs exist
if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = "unsloth_output"
if "MERGED_DIR" not in globals():
    MERGED_DIR = os.path.join(OUTPUT_DIR, "merged")
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "lora_adapter")
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

# 2) Save LoRA adapter weights (small + reusable)
#    Works for PEFT/Unsloth models
MODEL_OBJ.save_pretrained(ADAPTER_DIR)
if "tokenizer" in globals():
    tokenizer.save_pretrained(ADAPTER_DIR)
print("✅ Saved LoRA adapter to:", ADAPTER_DIR)

# 3) Merge LoRA into base weights (standalone checkpoint)
#    If you previously called merge_and_unload(), reuse it, else do it now.
if "merged_model" not in globals():
    try:
        merged_model = MODEL_OBJ.merge_and_unload()
    except AttributeError:
        # Some envs keep the merge on the PEFT wrapper only—try getattr on underlying .base_model
        base = getattr(MODEL_OBJ, "base_model", None)
        if base is None or not hasattr(base, "merge_and_unload"):
            raise SystemExit("❌ Could not find `merge_and_unload()` on the model. Ensure you're saving a LoRA/PEFT model.")
        merged_model = base.merge_and_unload()

# Try Unsloth helper first, fall back to HF save_pretrained if signature differs
try:
    from unsloth import unsloth_save_model
    try:
        # Signature variant 1: (model, save_directory)
        unsloth_save_model(merged_model, MERGED_DIR)
    except TypeError:
        # Signature variant 2: keyword args
        unsloth_save_model(model=merged_model, save_directory=MERGED_DIR)
    print("✅ Saved merged model via unsloth_save_model →", MERGED_DIR)
except Exception as e:
    print("⚠️ unsloth_save_model unavailable/incompatible:", repr(e))
    print("→ Falling back to Transformers .save_pretrained()")
    merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
    print("✅ Saved merged model via .save_pretrained →", MERGED_DIR)

# 4) Always save tokenizer into merged dir too
if "tokenizer" in globals():
    tokenizer.save_pretrained(MERGED_DIR)
print("✅ Saved tokenizer to:", MERGED_DIR)

# 5) Helpful summary
print("\nSummary:")
print("  Adapter dir:", ADAPTER_DIR)
print("  Merged dir :", MERGED_DIR)


✅ Saved LoRA adapter to: smollm2_135m_lora_alpaca/lora_adapter
⚠️ unsloth_save_model unavailable/incompatible: TypeError("unsloth_save_model() missing 1 required positional argument: 'tokenizer'")
→ Falling back to Transformers .save_pretrained()
✅ Saved merged model via .save_pretrained → smollm2_135m_lora_alpaca_merged
✅ Saved tokenizer to: smollm2_135m_lora_alpaca_merged

Summary:
  Adapter dir: smollm2_135m_lora_alpaca/lora_adapter
  Merged dir : smollm2_135m_lora_alpaca_merged


In [13]:
from transformers import TextStreamer

infer_model, infer_tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MERGED_DIR,  # or adapter_dir to test the LoRA path
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = dtype,
    load_in_4bit    = False,
)
FastLanguageModel.for_inference(infer_model)

messages = [
    {"role": "user", "content": "Give me a short Python function that reverses a string."}
]
prompt = infer_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = infer_tokenizer([prompt], return_tensors="pt").to(infer_model.device)

streamer = TextStreamer(infer_tokenizer, skip_prompt=True, skip_special_tokens=True)
_ = infer_model.generate(**inputs, max_new_tokens=120, streamer=streamer)


==((====))==  Unsloth 2025.11.2: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Here is a simple Python function that reverses a string:

```python
def reverse_string(s):
    return s[::-1]
```

This function uses Python's slice notation to create a new string that is the reverse of the original string. The `[::-1]` slice means "start at the end of the string and end at position 0, move with the step -1", effectively reversing the string.
